# Arithmetic SAQ — Orchestrator Fine-Tuning (Qwen2.5-0.5B-Instruct)

Full-parameter SFT that trains the model as a **task router**: given a time/date arithmetic question, predict which of the 9 task categories it belongs to.

Categories: `Hour Adjustment (24h)` · `Hour Adjustment (12h)` · `Year Shift` · `Month Shift` · `Date Computation` · `Week Identification` · `Time Zone Conversion` · `Time Computation` · `Application`

Split: **8 : 1 : 1** (train / val / test), stratified by category.  
Primary metric: **exact-match accuracy** on the predicted category label.  
Features:
- Checkpoint saved every N steps; auto-resume from last checkpoint
- SIGINT / SIGTERM handled gracefully — saves an emergency checkpoint before stopping
- `BestModelCallback` saves weights whenever val EM improves

## Section 0: Setup

In [1]:
# Cell 0.1 — Dependency check
import importlib, subprocess, sys

REQUIRED = [
    ("transformers", "transformers>=4.40"),
    ("datasets",     "datasets"),
    ("trl",          "trl>=0.8"),
    ("accelerate",   "accelerate"),
    ("sklearn",      "scikit-learn"),
]
missing_pkgs = [pkg for mod, pkg in REQUIRED if importlib.util.find_spec(mod) is None]
if missing_pkgs:
    print(f"Installing: {missing_pkgs}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs])
print("All dependencies present.")

All dependencies present.


In [2]:
# Cell 0.2 — Imports
import os, sys, json, signal, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainerCallback,
    TrainerControl,
    TrainerState,
    TrainingArguments,
)
from trl import SFTConfig, SFTTrainer

print(f"PyTorch:      {torch.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info(0)
    print(f"VRAM:         {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

c:\Users\ADMIN\miniconda3\envs\mt5\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch:      2.11.0+cu126
CUDA:         True
GPU:          NVIDIA GeForce RTX 4050 Laptop GPU
VRAM:         5.3 GB free / 6.4 GB total


In [3]:
# Cell 0.3 — Config
# ── Paths ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(os.getcwd()).resolve()
REPO_ROOT    = NOTEBOOK_DIR.parent   # notebooks/ -> repo root

DATA_MAIN  = str(REPO_ROOT / "src" / "data" / "arithmetic" / "arithmetic_saq.csv")
DATA_SHOTS = str(REPO_ROOT / "src" / "data" / "arithmetic" / "arithmetic_shots_saq.csv")
OUTPUT_DIR = str(REPO_ROOT / "outputs" / "arithmetic_orchestrator_qwen05b")

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# ── Task categories the orchestrator must predict ─────────────────────────────
CATEGORIES = [
    "Hour Adjustment (24h)",
    "Hour Adjustment (12h)",
    "Year Shift",
    "Month Shift",
    "Date Computation",
    "Week Identification",
    "Time Zone Conversion",
    "Time Computation",
    "Application",
]
CATEGORY_LIST_STR = "\n".join(f"- {c}" for c in CATEGORIES)

SYSTEM_PROMPT = (
    "You are an orchestrator for time and date arithmetic tasks. "
    "Given a question, classify it into exactly one of the following categories:\n"
    f"{CATEGORY_LIST_STR}\n\n"
    "Output only the category name, nothing else."
)

# ── Split ────────────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1
SEED        = 42

# ── Training ─────────────────────────────────────────────────────────────────
NUM_EPOCHS    = 5
BATCH_SIZE    = 8
EVAL_BATCH    = 16
GRAD_ACCUM    = 4       # effective batch = BATCH_SIZE * GRAD_ACCUM
LR            = 2e-5
MAX_SEQ_LEN   = 256
WARMUP_RATIO  = 0.05
WEIGHT_DECAY  = 0.01
SAVE_STEPS    = 50
LOGGING_STEPS = 10

# ── Generation ───────────────────────────────────────────────────────────────
MAX_NEW_TOKENS = 16     # category names are short

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Data main:   {DATA_MAIN}")
print(f"Data shots:  {DATA_SHOTS}")
print(f"\nOrchestrator categories ({len(CATEGORIES)}):")
for c in CATEGORIES:
    print(f"  {c}")

Output dir:  F:\temporal\outputs\arithmetic_orchestrator_qwen05b
Data main:   F:\temporal\src\data\arithmetic\arithmetic_saq.csv
Data shots:  F:\temporal\src\data\arithmetic\arithmetic_shots_saq.csv

Orchestrator categories (9):
  Hour Adjustment (24h)
  Hour Adjustment (12h)
  Year Shift
  Month Shift
  Date Computation
  Week Identification
  Time Zone Conversion
  Time Computation
  Application


## Section 1: Data Loading & Splitting

In [4]:
# Cell 1.1 — Load & combine CSVs
df_main  = pd.read_csv(DATA_MAIN)
df_shots = pd.read_csv(DATA_SHOTS)

df = pd.concat([df_main, df_shots], ignore_index=True)
df = df.dropna(subset=["Question", "Answer"]).copy()
df["Question"] = df["Question"].astype(str).str.strip()
df["Answer"]   = df["Answer"].astype(str).str.strip()
df["Category"] = df["Category"].astype(str).str.strip()

# Remove duplicates (shots csv shares some examples with the main csv)
before = len(df)
df = df.drop_duplicates(subset=["Question", "Answer"]).reset_index(drop=True)
print(f"Main: {len(df_main):,}  Shots: {len(df_shots):,}  After dedup: {len(df):,} (removed {before-len(df)})")
print(f"\nCategory distribution:")
print(df["Category"].value_counts().to_string())

Main: 15,584  Shots: 45  After dedup: 15,629 (removed 0)

Category distribution:
Category
Date Computation         6000
Application              2042
Hour Adjustment (24h)    1500
Hour Adjustment (12h)    1500
Week Identification      1497
Year Shift               1470
Time Computation          980
Time Zone Conversion      500
Month Shift               140


In [5]:
# Cell 1.2 — Stratified 8:1:1 split
from sklearn.model_selection import train_test_split

random.seed(SEED)
np.random.seed(SEED)

# 80% train, 20% temp
train_df, temp_df = train_test_split(
    df,
    test_size=(VAL_RATIO + TEST_RATIO),
    random_state=SEED,
    stratify=df["Category"],
)
# 50/50 on the 20% temp -> 10% val, 10% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df["Category"],
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

n = len(df)
print(f"Total: {n:,}")
print(f"Train: {len(train_df):,}  ({len(train_df)/n:.0%})")
print(f"Val:   {len(val_df):,}  ({len(val_df)/n:.0%})")
print(f"Test:  {len(test_df):,}  ({len(test_df)/n:.0%})")

print("\nCategory distribution per split:")
for name, sdf in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"  {name}: {sdf['Category'].value_counts().to_dict()}")

Total: 15,629
Train: 12,503  (80%)
Val:   1,563  (10%)
Test:  1,563  (10%)

Category distribution per split:
  train: {'Date Computation': 4800, 'Application': 1633, 'Hour Adjustment (12h)': 1200, 'Hour Adjustment (24h)': 1200, 'Week Identification': 1198, 'Year Shift': 1176, 'Time Computation': 784, 'Time Zone Conversion': 400, 'Month Shift': 112}
  val: {'Date Computation': 600, 'Application': 204, 'Week Identification': 150, 'Hour Adjustment (24h)': 150, 'Hour Adjustment (12h)': 150, 'Year Shift': 147, 'Time Computation': 98, 'Time Zone Conversion': 50, 'Month Shift': 14}
  test: {'Date Computation': 600, 'Application': 205, 'Hour Adjustment (12h)': 150, 'Hour Adjustment (24h)': 150, 'Week Identification': 149, 'Year Shift': 147, 'Time Computation': 98, 'Time Zone Conversion': 50, 'Month Shift': 14}


## Section 2: Tokenizer & Model

In [6]:
# Cell 2.1 — Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # SFTTrainer requires right-padding

# ── Chat-format helpers ───────────────────────────────────────────────────────
def format_for_sft(row):
    """Training sample: user=Question, assistant=Category (the routing label)."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": row["Question"]},
        {"role": "assistant", "content": row["Category"]},   # <-- label to predict
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

def build_inference_prompt(row):
    """Inference prompt: no answer; model must predict the category."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": row["Question"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Sanity check
sample = train_df.iloc[0]
print(f"Gold category: {sample['Category']!r}")
print("\n=== Training format sample ===")
print(format_for_sft(sample))
print("\n=== Inference prompt sample ===")
print(build_inference_prompt(sample))

Gold category: 'Hour Adjustment (12h)'

=== Training format sample ===
<|im_start|>system
You are an orchestrator for time and date arithmetic tasks. Given a question, classify it into exactly one of the following categories:
- Hour Adjustment (24h)
- Hour Adjustment (12h)
- Year Shift
- Month Shift
- Date Computation
- Week Identification
- Time Zone Conversion
- Time Computation
- Application

Output only the category name, nothing else.<|im_end|>
<|im_start|>user
What is 06:05 AM - 03:53?<|im_end|>
<|im_start|>assistant
Hour Adjustment (12h)<|im_end|>


=== Inference prompt sample ===
<|im_start|>system
You are an orchestrator for time and date arithmetic tasks. Given a question, classify it into exactly one of the following categories:
- Hour Adjustment (24h)
- Hour Adjustment (12h)
- Year Shift
- Month Shift
- Date Computation
- Week Identification
- Time Zone Conversion
- Time Computation
- Application

Output only the category name, nothing else.<|im_end|>
<|im_start|>user
What 

In [7]:
# Cell 2.2 — Load model (full fine-tuning, no LoRA)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,} (100% — full fine-tuning)")

if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1e9
    resv  = torch.cuda.memory_reserved() / 1e9
    print(f"GPU memory: {alloc:.2f} GB allocated / {resv:.2f} GB reserved")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 414.01it/s]


Total params:     494,032,768
Trainable params: 494,032,768 (100% — full fine-tuning)
GPU memory: 0.99 GB allocated / 1.02 GB reserved


## Section 3: Training

In [8]:
# Cell 3.1 — Interrupt handler (SIGINT / SIGTERM)
_interrupted = False

def _signal_handler(signum, frame):
    global _interrupted
    _interrupted = True
    print(f"\n[Signal {signum}] Interrupt received — will save checkpoint after current step.")

signal.signal(signal.SIGINT,  _signal_handler)
signal.signal(signal.SIGTERM, _signal_handler)
print("Signal handlers registered (SIGINT, SIGTERM).")

Signal handlers registered (SIGINT, SIGTERM).


In [9]:
# Cell 3.2 — Exact-match evaluation helper (gold = Category label)
def evaluate_exact_match(eval_model, eval_df, batch_size=EVAL_BATCH, verbose_n=3):
    """
    Greedy-generate the predicted category and compare to the true Category.
    Returns (overall_em: float, results_df: pd.DataFrame).
    """
    eval_model.eval()
    rows = eval_df.to_dict("records")
    preds, golds, questions = [], [], []

    for i in range(0, len(rows), batch_size):
        batch   = rows[i : i + batch_size]
        prompts = [build_inference_prompt(r) for r in batch]

        enc = tokenizer(
            prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_SEQ_LEN,
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            out_ids = eval_model.generate(
                **enc,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_len = enc["input_ids"].shape[1]
        for j, (row, out) in enumerate(zip(batch, out_ids)):
            pred  = tokenizer.decode(out[prompt_len:], skip_special_tokens=True).strip()
            gold  = str(row["Category"]).strip()   # routing target
            match = pred.lower() == gold.lower()
            preds.append(pred)
            golds.append(gold)
            questions.append(row["Question"])
            if i == 0 and j < verbose_n:
                status = "OK" if match else "FAIL"
                print(f"  [{j+1}] [{status}] gold={gold!r}  pred={pred!r}")
                print(f"       Q: {row['Question'][:70]}")

    eval_model.train()

    matches = [p.lower() == g.lower() for p, g in zip(preds, golds)]
    overall = sum(matches) / len(matches) if matches else 0.0

    results_df = pd.DataFrame({
        "Question": questions,
        "Gold":     golds,
        "Pred":     preds,
        "Match":    matches,
        "Category": golds,   # same as Gold; kept for groupby convenience
    })
    return overall, results_df

print("evaluate_exact_match() defined.")

evaluate_exact_match() defined.


In [10]:
# Cell 3.3 — BestModelCallback: tracks highest val EM and saves weights; also handles interrupts
class BestModelCallback(TrainerCallback):
    """
    Runs generation-based exact-match evaluation on the validation set at the
    end of every epoch. Saves model + tokenizer when EM improves.
    Also checks the interrupt flag and stops training gracefully.
    """

    def __init__(self, val_df, best_dir, eval_every_n_epochs=1):
        self.val_df  = val_df
        self.dir     = Path(best_dir)
        self.every   = eval_every_n_epochs
        self.best_em = -1.0
        self.history = []

    def on_epoch_end(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        model=None,
        **kwargs,
    ):
        epoch = round(state.epoch)
        if epoch % self.every != 0:
            return

        print(f"\n[BestModelCallback] epoch={epoch} — evaluating val exact-match...")
        em, results = evaluate_exact_match(model, self.val_df)

        cat_em = results.groupby("Category")["Match"].mean().to_dict()
        entry  = {"epoch": epoch, "val_em": em,
                  **{f"em_{k}": v for k, v in cat_em.items()}}
        self.history.append(entry)

        print(f"[BestModelCallback] val_em={em:.4f}  (best={self.best_em:.4f})")

        if em > self.best_em:
            self.best_em = em
            self.dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(str(self.dir))
            tokenizer.save_pretrained(str(self.dir))
            with open(self.dir / "best_metric.json", "w") as f:
                json.dump(entry, f, indent=2)
            print(f"[BestModelCallback] New best ({em:.4f}) — saved to {self.dir}")

        # Persist training curve
        with open(Path(OUTPUT_DIR) / "em_history.json", "w") as f:
            json.dump(self.history, f, indent=2)

    def on_step_end(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        **kwargs,
    ):
        if _interrupted:
            print("[BestModelCallback] Interrupt flag set — requesting training stop after this step.")
            control.should_save          = True
            control.should_training_stop = True


best_dir = Path(OUTPUT_DIR) / "best_model"
bm_cb    = BestModelCallback(val_df, best_dir)
print(f"BestModelCallback ready. Best model dir: {best_dir}")

BestModelCallback ready. Best model dir: F:\temporal\outputs\arithmetic_orchestrator_qwen05b\best_model


In [11]:
# Cell 3.4 — Build HuggingFace Datasets
def df_to_hf_dataset(df, shuffle=False):
    texts = [format_for_sft(row) for _, row in df.iterrows()]
    ds = Dataset.from_dict({"text": texts})
    return ds.shuffle(seed=SEED) if shuffle else ds

train_hf = df_to_hf_dataset(train_df, shuffle=True)
val_hf   = df_to_hf_dataset(val_df)

print(f"Train HF: {len(train_hf):,} samples")
print(f"Val   HF: {len(val_hf):,} samples")
print(f"\nSample text (first 300 chars):\n{train_hf[0]['text'][:300]}")

Train HF: 12,503 samples
Val   HF: 1,563 samples

Sample text (first 300 chars):
<|im_start|>system
You are an orchestrator for time and date arithmetic tasks. Given a question, classify it into exactly one of the following categories:
- Hour Adjustment (24h)
- Hour Adjustment (12h)
- Year Shift
- Month Shift
- Date Computation
- Week Identification
- Time Zone Conversion
- Time


In [12]:
# Cell 3.5 — SFTTrainer configuration
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

sft_cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    bf16=use_bf16,
    fp16=use_fp16,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,          # keep only the 3 most recent step-checkpoints
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    load_best_model_at_end=False,   # BestModelCallback handles this
    dataloader_num_workers=0,
    report_to="none",
    # SFT-specific
    # max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_cfg,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    processing_class=tokenizer,
    callbacks=[bm_cb],
)
print("SFTTrainer ready.")
print(f"Steps per epoch: {len(trainer.get_train_dataloader()):,}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Tokenizing eval dataset: 100%|██████████| 1563/1563 [00:00<00:00, 1616.28 examples/s]


SFTTrainer ready.
Steps per epoch: 1,563


In [ ]:
# Cell 3.6 — Run training (auto-resumes from last checkpoint)
# Find latest checkpoint in OUTPUT_DIR
ckpt_dir  = Path(OUTPUT_DIR)
last_ckpt = None
if ckpt_dir.exists():
    ckpts = sorted(
        [p for p in ckpt_dir.glob("checkpoint-*")
         if p.name.split("-")[-1].isdigit()],
        key=lambda p: int(p.name.split("-")[-1]),
    )
    if ckpts:
        last_ckpt = str(ckpts[-1])
        print(f"Resuming from checkpoint: {last_ckpt}")
    else:
        print("No prior checkpoint found — starting fresh.")

print("Starting training...")
t0 = time.time()

try:
    trainer.train(resume_from_checkpoint=last_ckpt)
except KeyboardInterrupt:
    # Fallback in case signal handler is bypassed in some environments
    emergency_dir = os.path.join(OUTPUT_DIR, "emergency_checkpoint")
    print(f"\nKeyboardInterrupt — saving emergency checkpoint to {emergency_dir}")
    trainer.save_model(emergency_dir)
    tokenizer.save_pretrained(emergency_dir)

elapsed = time.time() - t0
print(f"\nTraining finished in {elapsed/60:.1f} min.")
print(f"Best val exact-match: {bm_cb.best_em:.4f}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Resuming from checkpoint: F:\temporal\outputs\arithmetic_orchestrator_qwen05b\checkpoint-200
Starting training...


[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Step,Training Loss,Validation Loss
250,0.146100,0.145603
300,0.144908,0.145714
350,0.145940,0.144737
400,0.144511,0.144370
450,0.143897,0.144339
500,0.143180,0.144428
550,0.144823,0.144628
600,0.142042,0.144275
650,0.142750,0.144362
700,0.141962,0.144163


Writing model shards: 100%|██████████| 1/1 [00:15<00:00, 15.07s/it]


...

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, bu

[BestModelCallback] val_em=0.0947  (best=0.0947)


Writing model shards: 100%|██████████| 1/1 [00:14<00:00, 14.26s/it]



KeyboardInterrupt — saving emergency checkpoint to F:\temporal\outputs\arithmetic_orchestrator_qwen05b\emergency_checkpoint


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Section 4: Evaluation

In [ ]:
# Cell 4.1 — Validation set evaluation (current / last model)
print("=" * 50)
print("Validation Set — Current Model")
print("=" * 50)

val_em, val_results = evaluate_exact_match(model, val_df, verbose_n=5)
print(f"\nVal Exact Match: {val_em:.4f}  ({int(val_em*len(val_df))}/{len(val_df)})")

print("\nPer-Category EM:")
cat_stats = val_results.groupby("Category")["Match"].agg(["mean", "count"])
for cat, row in cat_stats.sort_values("mean", ascending=False).iterrows():
    print(f"  {cat:<35} {row['mean']:.3f}  ({int(row['mean']*row['count'])}/{int(row['count'])})")

In [ ]:
# Cell 4.2 — Test set evaluation (current / last model)
print("=" * 50)
print("Test Set — Current Model")
print("=" * 50)

test_em, test_results = evaluate_exact_match(model, test_df, verbose_n=5)
print(f"\nTest Exact Match: {test_em:.4f}  ({int(test_em*len(test_df))}/{len(test_df)})")

print("\nPer-Category EM:")
cat_stats = test_results.groupby("Category")["Match"].agg(["mean", "count"])
for cat, row in cat_stats.sort_values("mean", ascending=False).iterrows():
    print(f"  {cat:<35} {row['mean']:.3f}  ({int(row['mean']*row['count'])}/{int(row['count'])})")

results_path = Path(OUTPUT_DIR) / "test_results_current.csv"
test_results.to_csv(results_path, index=False)
print(f"\nSaved → {results_path}")

In [ ]:
# Cell 4.3 — Load best checkpoint and run final evaluation
best_model_dir = Path(OUTPUT_DIR) / "best_model"

if best_model_dir.exists() and (best_model_dir / "best_metric.json").exists():
    with open(best_model_dir / "best_metric.json") as f:
        best_info = json.load(f)
    print(f"Best checkpoint info: {best_info}")

    # Load the full saved model directly
    best_model = AutoModelForCausalLM.from_pretrained(
        str(best_model_dir),
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    best_model.eval()

    print("\n" + "=" * 50)
    print("Test Set — Best Model")
    print("=" * 50)
    best_test_em, best_test_results = evaluate_exact_match(best_model, test_df, verbose_n=5)
    print(f"\nTest Exact Match (best model): {best_test_em:.4f}  ({int(best_test_em*len(test_df))}/{len(test_df)})")

    print("\nPer-Category EM:")
    cat_stats = best_test_results.groupby("Category")["Match"].agg(["mean", "count"])
    for cat, row in cat_stats.sort_values("mean", ascending=False).iterrows():
        print(f"  {cat:<35} {row['mean']:.3f}  ({int(row['mean']*row['count'])}/{int(row['count'])})")

    best_path = Path(OUTPUT_DIR) / "test_results_best.csv"
    best_test_results.to_csv(best_path, index=False)
    print(f"\nSaved → {best_path}")
else:
    print("No best_model directory found — skipping.")

In [ ]:
# Cell 4.4 — Training curve (val EM over epochs)
history_path = Path(OUTPUT_DIR) / "em_history.json"
if history_path.exists():
    with open(history_path) as f:
        history = json.load(f)

    import matplotlib.pyplot as plt

    epochs  = [h["epoch"] for h in history]
    val_ems = [h["val_em"] for h in history]

    plt.figure(figsize=(8, 4))
    plt.plot(epochs, val_ems, marker="o", label="val_em")
    plt.xlabel("Epoch")
    plt.ylabel("Exact Match")
    plt.title("Validation Exact Match per Epoch")
    plt.ylim(0, 1)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    fig_path = Path(OUTPUT_DIR) / "em_history.png"
    plt.savefig(fig_path, dpi=100)
    plt.show()
    print(f"Saved → {fig_path}")
else:
    print("No em_history.json found — run training first.")

## Section 5: Inference Demo

In [ ]:
# Cell 5.1 — Orchestrator routing demo
DEMO_QUESTIONS = [
    "What is 14:30 + 03:45?",
    "What is 09:20 PM + 04:50?",
    "Which month comes 4 months after September?",
    "If you add 60 days to the date 03-01-2024, what will be the new date?",
    "Which year comes 12 years after 2008?",
    "Convert 135 minutes into hours.",
    "In which week of year 2024 does the date 07-04-2024 occur?",
    "If it's 03 PM on Jan 1, 2020 in US/Eastern, what's the date and time in UTC?",
    "If a girl was 3 years old when she joined school and now she is 10 years 2 months old, for how long has she been in school?",
]

infer_model = best_model if "best_model" in dir() else model
infer_model.eval()

print(f"{'Question':<70} {'Predicted Category'}")
print("-" * 110)
for question in DEMO_QUESTIONS:
    fake_row = {"Question": question, "Category": ""}
    prompt   = build_inference_prompt(fake_row)
    enc      = tokenizer(prompt, return_tensors="pt",
                         truncation=True, max_length=MAX_SEQ_LEN).to(device)
    with torch.no_grad():
        out_ids = infer_model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    pred = tokenizer.decode(out_ids[0, enc["input_ids"].shape[1]:],
                            skip_special_tokens=True).strip()
    print(f"{question[:68]:<70} {pred}")